# Chapter 10: From Spirit LM to Moshi and CSM

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [From Spirit LM to Moshi and CSM](https://books.sugiv.fyi/voice_applications/speech-to-speech.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## Three steps toward one model

## Moshi takes Halfwheel's calls

In [ ]:
import json
import sys

import numpy as np
from IPython.display import Audio, display

sys.path.insert(0, "sim")
from wav import load  # Chapter 2's WAV reader

moshi = json.load(open("data/ch10/moshi.json"))


def utterances(pieces, gap=0.6):
    """Group Moshi's words into utterances: a new one starts after 0.6 s without a word."""
    words = [t for t, p in pieces if p.strip()]
    out, start, last = [], None, None
    for t in words:
        if last is None or t - last > gap:
            if start is not None:
                out.append((start, last))
            start = t
        last = t
    return out + ([(start, last)] if start is not None else [])


runs = []
for cid, c in moshi["calls"].items():
    spoke = c["segments"]
    start, end = spoke[0][0], spoke[-1][1]
    pauses = [(spoke[k][1], spoke[k + 1][0]) for k in range(len(spoke) - 1)]
    for s in c["samples"]:
        u = utterances(s["pieces"])
        after = [a - end for a, _ in u if a >= end]
        runs.append({"call": cid, "kind": c["kind"], "text": s["text"],
                     "greeted": any(a < start for a, _ in u),
                     "over": any(a < end and b > start + 0.3 for a, b in u),
                     "in_pause": any(pa + 0.2 < a < pb for a, _ in u for pa, pb in pauses),
                     "reply_after": after[0] if after else None})

print(f"{'':10s} runs  greeted  talked over  reply within 2 s  no reply  started in a pause")
for kind in ["opener", "hesitant"]:
    r = [x for x in runs if x["kind"] == kind]
    print(f"{kind:10s} {len(r):4d} {sum(x['greeted'] for x in r):8d} {sum(x['over'] for x in r):11d} "
          f"{sum(x['reply_after'] is not None and x['reply_after'] <= 2 for x in r):17d} "
          f"{sum(x['reply_after'] is None for x in r):9d} {sum(x['in_pause'] for x in r):18d}")
waits = [x["reply_after"] for x in runs if x["kind"] == "opener" and x["reply_after"] is not None]
print(f"\nopeners: median wait for a new reply {np.median(waits):.1f} s")

## What it says

In [ ]:
by_call = {}
for x in runs:
    by_call.setdefault(x["call"], []).append(x)
for call, sample in [("c01_0", 0), ("c03_0", 2), ("c04_0", 0), ("c02_0", 1)]:
    line = by_call[call][sample]["text"].replace("\n", " ")
    print(f"{call}, run {sample + 1}: {line[:190]}\n")
for name in ["moshi_c01_0", "moshi_c04_0"]:
    x, sr = load(f"data/ch10/listen/{name}.wav")
    display(Audio(x, rate=sr))

## Conversation as context: CSM

In [ ]:
csm = json.load(open("data/ch10/csm.json"))
measures = [("f0_median", "median pitch (Hz)"), ("f0_range_semitones", "pitch range (semitones)"),
            ("words_per_s", "words per second"), ("loudness_db", "loudness (dB)"), ("seconds", "length (s)")]
means = {mood: {k: np.mean([c[k] for c in csm["clips"].values() if c["mood"] == mood]) for k, _ in measures}
         for mood in ["calm", "angry"]}
print(f"{'':26s} {'after a calm caller':>20s} {'after an angry one':>20s}")
for k, label in measures:
    print(f"{label:26s} {means['calm'][k]:20.2f} {means['angry'][k]:20.2f}")
for mood in ["calm", "angry"]:
    print(f"\nafter the {mood} caller: \"{csm['replies'][0]}\"")
    x, sr = load(f"data/ch10/listen/csm_{mood}_r0_0.wav")
    display(Audio(x, rate=sr))